In [ ]:
"""Day 27 / Session 3 - Lab Assignment 1: Classifier-Free Guidance - Steering Diffusion Toward the Prompt."""
import numpy as np


def cfg_combine(eps_uncond, eps_cond, w):
    u = np.asarray(eps_uncond, dtype=float)
    c = np.asarray(eps_cond, dtype=float)
    return u + w * (c - u)                                   # w=0 -> unconditional, w=1 -> conditional, w>1 extrapolates


def conditional_direction(eps_uncond, eps_cond):
    return np.asarray(eps_cond, dtype=float) - np.asarray(eps_uncond, dtype=float)


def guidance_projection(eps_uncond, eps_cond, w):
    d = conditional_direction(eps_uncond, eps_cond).ravel()
    norm = np.linalg.norm(d)
    if norm == 0:
        return 0.0
    guided = cfg_combine(eps_uncond, eps_cond, w).ravel()
    return float(np.dot(guided, d / norm))


def guidance_alignment(eps_uncond, eps_cond, w):
    d = conditional_direction(eps_uncond, eps_cond).ravel()
    g = cfg_combine(eps_uncond, eps_cond, w).ravel()
    nd, ng = np.linalg.norm(d), np.linalg.norm(g)
    if nd == 0 or ng == 0:
        return 0.0
    return float(np.dot(g, d) / (nd * ng))


def alignment_curve(eps_uncond, eps_cond, ws):
    return np.array([guidance_alignment(eps_uncond, eps_cond, w) for w in ws])
